# Phase 1: Multi-Horizon Data Collection

This notebook serves as the foundation for our Hybrid Temporal Forecaster. We are building a dual-horizon dataset:
1. **Strategic Daily Data (20 Years)**: To capture long-term market regimes and economic memory.
2. **Tactical Hourly Data (2 Years)**: To capture intra-day volatility and short-term forecasting patterns.

By combining these perspectives, we can evaluate how broad regimes influence immediate tactical opportunities.

## 1. Imports and Reproducibility

As always, we establish a global seed of `25` to ensure that any stochastic data operations remain perfectly reproducible.

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import random
import os
from datetime import datetime, timedelta

SEED = 25
np.random.seed(SEED)
random.seed(SEED)

print(f"Global seed established: {SEED}")

Global seed established: 25


## 2. Configuration and Horizons

We target a universe of 5 assets (AAPL, MSFT, JPM, SPY, GLD). Due to API limitations, hourly data is restricted to the last 730 days, whereas daily data stretches back 20 years.

In [2]:
TICKERS = ["AAPL", "MSFT", "JPM", "SPY", "GLD"]
END_DATE = datetime.now().strftime("%Y-%m-%d")

# Daily configuration (~20 years)
DAILY_START_DATE = (datetime.now() - timedelta(days=365*20)).strftime("%Y-%m-%d")
DAILY_DIR = "../data/raw/daily"

# Hourly configuration (~2 years)
HOURLY_START_DATE = (datetime.now() - timedelta(days=729)).strftime("%Y-%m-%d")
HOURLY_DIR = "../data/raw/hourly"

os.makedirs(DAILY_DIR, exist_ok=True)
os.makedirs(HOURLY_DIR, exist_ok=True)

print(f"Daily horizon: {DAILY_START_DATE} to {END_DATE}")
print(f"Hourly horizon: {HOURLY_START_DATE} to {END_DATE}")

Daily horizon: 2006-03-27 to 2026-03-22
Hourly horizon: 2024-03-23 to 2026-03-22


## 3. Implementation of the Ingestion Pipeline

We define a unified downloader that handles different intervals (1d, 1h) and stores them in organized subdirectories.

In [3]:
def fetch_data(tickers, start, end, interval, target_dir):
    for ticker in tickers:
        print(f"Processing {ticker} at {interval} interval...")
        df = yf.download(ticker, start=start, end=end, interval=interval)
        if not df.empty:
            path = os.path.join(target_dir, f"{ticker}.csv")
            df.to_csv(path)
            print(f"  -> Saved to {path} ({len(df)} rows)")
        else:
            print(f"  !! Warning: Failed to fetch {ticker}")

print("Starting Strategic Data Collection (Daily)...")
fetch_data(TICKERS, DAILY_START_DATE, END_DATE, "1d", DAILY_DIR)

print("\nStarting Tactical Data Collection (Hourly)...")
fetch_data(TICKERS, HOURLY_START_DATE, END_DATE, "1h", HOURLY_DIR)

Starting Strategic Data Collection (Daily)...
Processing AAPL at 1d interval...


[*********************100%***********************]  1 of 1 completed

  -> Saved to ../data/raw/daily/AAPL.csv (5028 rows)
Processing MSFT at 1d interval...


[*********************100%***********************]  1 of 1 completed

  -> Saved to ../data/raw/daily/MSFT.csv (5028 rows)
Processing JPM at 1d interval...


...

  -> Saved to ../data/raw/hourly/JPM.csv (3459 rows)
Processing SPY at 1h interval...


[*********************100%***********************]  1 of 1 completed

  -> Saved to ../data/raw/hourly/SPY.csv (3459 rows)
Processing GLD at 1h interval...


[*********************100%***********************]  1 of 1 completed

  -> Saved to ../data/raw/hourly/GLD.csv (3459 rows)


## 4. Final Verification and Storage Audit

We conclude with a quick cross-reference to ensure that all assets across both horizons are properly cached for Phase 2.

In [4]:
audit = []
for horizon, path in [("Daily", DAILY_DIR), ("Hourly", HOURLY_DIR)]:
    for ticker in TICKERS:
        file_path = os.path.join(path, f"{ticker}.csv")
        if os.path.exists(file_path):
            df = pd.read_csv(file_path)
            audit.append({
                "Horizon": horizon,
                "Ticker": ticker,
                "Rows": len(df),
                "Start": df.iloc[0, 0],
                "End": df.iloc[-1, 0]
            })

pd.DataFrame(audit)

,Horizon,Ticker,Rows,Start,End
0,Daily,AAPL,5030,Ticker,2026-03-20
1,Daily,MSFT,5030,Ticker,2026-03-20
2,Daily,JPM,5030,Ticker,2026-03-20
3,Daily,SPY,5030,Ticker,2026-03-20
4,Daily,GLD,5030,Ticker,2026-03-20
5,Hourly,AAPL,3462,Ticker,2026-03-20 19:30:00+00:00
6,Hourly,MSFT,3462,Ticker,2026-03-20 19:30:00+00:00
7,Hourly,JPM,3461,Ticker,2026-03-20 19:30:00+00:00
8,Hourly,SPY,3461,Ticker,2026-03-20 19:30:00+00:00
9,Hourly,GLD,3461,Ticker,2026-03-20 19:30:00+00:00
